# 语音到 CARLA 车辆控制：部署与验收手册

本 Notebook 是可随代码交接的执行说明。控制链路为：`音频 → voice_group → VoiceCommandAdapter → A FSM → B 横向 + C 纵向 → D 安全仲裁 → CARLA apply_control`。

首次验收先执行离线闭环；真实 CARLA 和 ASR 在后续单元按需启动。

## 1. 环境与边界

- 本机开发环境：Conda `carla`、Python 3.12、CARLA 0.9.16。
- 容器环境：CARLA 服务端与控制器分为两个容器；控制器镜像默认可不含语音依赖（`WITH_VOICE=0`）。
- 真实 ASR 需要 NVIDIA GPU、匹配 CUDA 的 PyTorch、FunASR/ModelScope 模型缓存；模型缓存必须和镜像 tar 一起归档。
- 初始 CARLA 运行器用 simulator ground truth 生成前车/信号状态，只用于验证控制闭环；替换 RGB/LiDAR 感知时必须保持 `PerceptionFrame` 的字段和帧对齐。

In [ ]:
# 在仓库根目录、已激活 conda carla 环境时执行
!python -m pytest integration/tests car_control_A/tests car_control_B/tests car_control_C/tests car_control_D/tests -q
!python -m integration.demo_offline

## 2. 核心接口与安全规则

1. `voice_group.pipeline.audio_to_command()` 输出原始语音 JSON；原样保留给 D 审计。
2. `integration.VoiceCommandAdapter` 使用 CARLA `now_s` 写入 A 的接收/过期时间；绝不把主机 `monotonic_ns` 与仿真秒混用。`SET_SPEED` 从 km/h 转为 m/s。
3. `ControlRuntime.step()` 每帧调用 B 的 `step_any(vehicle, route)`、C 的 `step(request, dt)`，将 C 控制量和 B 转向合成后交给 D。仅 D 的 `final_control` 可以下发。
4. 无效、歧义、低置信度或需要多模态决策的语音命令会进入确认/安全停车路径；不会伪造变道、靠边或避障已执行。

In [ ]:
from integration import ControlRuntime, PerceptionFrame
from car_control_A import RuntimeVehicleState
from car_control_A.routing import RouteReference
from car_control_B.pure_pursuit import PurePursuitController

runtime = ControlRuntime(PurePursuitController())
voice = {
    'schema_version':'1.0', 'command_id':'notebook-speed', 'source_text':'速度设为18公里每小时',
    'intent':'SET_SPEED', 'parameters':{'speed':18, 'unit':'km/h'},
    'asr_confidence':0.99, 'intent_confidence':0.99, 'confidence':0.99,
    'status':'valid', 'ambiguity_type':'NONE', 'confirm_required':False, 'errors':[], 'warnings':[]
}
runtime.submit_voice(voice, now_s=0.05)
vehicle = RuntimeVehicleState(1, 0.05, 0, 0, 0, 0, 0, '1')
scene = PerceptionFrame(1, 0.05)
route = RouteReference(((0,0), (10,0), (20,0)), 0.0, 5.0)
result = runtime.step(vehicle, scene, route, dt_s=0.05)
result.final_control, result.safety_reason

## 3. 启动真实 CARLA

先启动本机 CARLA 0.9.16，或使用 Docker 的 `carla` 服务。准备 `artifacts/command.json`，格式与 `voice_control/parameter.txt` 相同。先用 JSON 验证控制闭环，再用 `--audio` 启动真实 ASR。

In [ ]:
# Windows 本机：CARLA 服务端启动后执行
!python -m integration.carla_runner --host 127.0.0.1 --port 2000 --frames 200 --command-json artifacts/command.json

# 已安装语音依赖、模型已缓存时，改用音频输入
# !python -m integration.carla_runner --host 127.0.0.1 --port 2000 --frames 200 --audio command.wav

## 4. Docker 构建、启动与归档

Docker Desktop 已检测可用。完整步骤见 `docker/README.md`：

1. `Copy-Item docker/.env.example docker/.env`。
2. `./docker/scripts/verify-stack.ps1 -BuildController` 构建控制器镜像。
3. `docker compose --env-file docker/.env -f docker/compose.yaml up -d carla`。
4. 在 `.env` 的 `CONTROLLER_COMMAND` 设置 `python3 -m integration.carla_runner --host carla --port 2000 --command-json /workspace/artifacts/command.json`，然后启动 controller profile。
5. `./docker/scripts/export-images.ps1 -IncludeCarla` 生成离线 tar 与 SHA256；另行备份 Docker 卷 `model-cache` 中的 ASR 模型。

## 5. 结果验证清单

- 离线：全部 pytest 通过；`demo_offline` 输出 `safety_override=False`、油门大于零、油门与制动不同时非零。
- 安全：`EMERGENCY_STOP` 输出全制动；感知帧不一致或集成异常输出 `INTEGRATION_FAILURE` 和全制动。
- CARLA：日志中每帧仅有一次 `apply_control`，`frame` 与状态/场景帧相同；JSON 指令能完成直线定速/停车。
- 语音：记录 `t_audio_start_ns/t_asr_end_ns/t_intent_end_ns`，以同源 monotonic 时间计算 ASR/NLU 延迟；CARLA 过期时间只使用仿真秒。
- 容器：镜像 tar 的 SHA256 与交付记录一致；离线 `docker image load` 后可重跑离线 demo。